# Heart Disease — Model Training
Trains and compares models, saves the best as
`models/heart_disease_model.pkl`.

In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("xgboost not installed — skipping it.")


## 1. Load preprocessed train/test data

In [3]:
X_train = pd.read_csv("../../data/processed/heart_disease_X_train.csv")
X_test = pd.read_csv("../../data/processed/heart_disease_X_test.csv")
y_train = pd.read_csv("../../data/processed/heart_disease_y_train.csv").squeeze()
y_test = pd.read_csv("../../data/processed/heart_disease_y_test.csv").squeeze()


## 2. Define models

In [4]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

if HAS_XGB:
    models["XGBoost"] = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)


## 3. Train and evaluate

In [5]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba),
    })

results_df = pd.DataFrame(results).sort_values("ROC-AUC", ascending=False)
results_df


c:\Users\SADAB EHTESHAM\Desktop\m project\venv\Lib\site-packages\xgboost\training.py:200: UserWarning: [21:49:33] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,Model,Accuracy,Precision,Recall,F1,ROC-AUC
2,Random Forest,0.819672,0.761905,0.969697,0.853333,0.912338
0,Logistic Regression,0.803279,0.769231,0.909091,0.833333,0.869048
3,XGBoost,0.803279,0.756098,0.939394,0.837838,0.856061
1,Decision Tree,0.704918,0.702703,0.787879,0.742857,0.697511


## 4. Pick and confirm the best model

In [6]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
print(f"Best model: {best_model_name}")

y_pred_best = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)
print(cm)


Best model: Random Forest
[[18 10]
 [ 1 32]]


## 5. Save the best model

In [7]:
joblib.dump(best_model, "../../models/heart_disease_model.pkl")
print("Saved: models/heart_disease_model.pkl")


Saved: models/heart_disease_model.pkl


## Next step
Open **04_evaluation.ipynb**.